# XAI Cấp độ 2 — Giải thích kết quả xếp hạng của CLIP + AdaMF trên DB15K và FB15K-237

Notebook này giải thích **phương pháp mới của cấp độ 2** (AdaMF, LREC-COLING 2024, bản tích hợp truy hồi của nhóm,
*không* có MAT) trên **2 tập công khai, mỗi tập 3 seed** (2027, 2028, 2029). Chỉ đọc checkpoint/kết quả đã khóa của nhóm,
không huấn luyện lại, không chọn gì trên test.

Cách mô hình xếp hạng (đọc từ notebook huấn luyện của nhóm):

| | DB15K (`04b1`) | FB15K-237 (`04b2`) |
|---|---|---|
| Huấn luyện | loss KGC (RotatE + adversarial sigmoid) trên bộ ba train | loss truy hồi query–ảnh (cross-entropy), bộ ba train chỉ đi vào qua truy vấn |
| Truy vấn gián tiếp `(anchor, r, ?)` | điểm đồ thị = `margin − d_RotatE(anchor, r, e)` của thực thể `e` gắn với ảnh | như DB15K, chia cho `dim`; ảnh nhiều thực thể lấy max |
| Truy vấn trực tiếp | giữ nguyên CLIP | `cos(chiếu(query), joint(e))` |
| Điểm cuối | `α·z(CLIP) + (1−α)·z(đồ thị)` trên top-1000 CLIP, α chọn trên validation | như DB15K |

`joint(e)` là tổng có trọng số attention của 3 phương thức: **cấu trúc** (embedding học từ KG), **ảnh**, **văn bản**.

| Bước | Nội dung |
|---|---|
| C0 | Cấu hình, tìm dữ liệu trên Drive |
| C1 | Tái lập kết quả test từ checkpoint cho 6 run, đối chiếu với `test_query_ranks.csv` của nhóm |
| C2 | Đồ thị cứu / làm hại bao nhiêu truy vấn (trực tiếp, head, tail) |
| C3 | Bộ 20 truy vấn đánh giá mỗi tập (10 trực tiếp + 10 gián tiếp), báo cáo tách nhóm |
| C4 | Giải thích tầng E1 (thực thể quyết định) + E2 (đóng góp từng phương thức, kiểm bằng che phương thức) |
| C5 | Giải thích tầng E3 (bộ ba huấn luyện làm mô hình tin, nối thành đường đi từ anchor) — học lại embedding kiểu Kelpie |
| C6 | Chọn số bộ ba K trên validation |
| C7 | Chọn ca theo luật cố định: 5 thành công + 5 thất bại mỗi tập |
| C8 | Hình và phân tích ca |
| C9 | Kiểm chứng giải thích đến từ cơ chế xếp hạng: bỏ bằng chứng so với bỏ ngẫu nhiên, 3 seed |
| C10 | Demo: nhập truy vấn → top-k + đồ thị giải thích tương tác |

**Chạy:** Colab GPU → Run all. Chỉ cần Drive có thư mục `MLGs - Project` (shortcut trong My Drive cũng được).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 1, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 2, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 3, globals())


## C1. Tái lập kết quả test từ checkpoint (6 run)

Mã hóa truy vấn bằng đúng CLIP text encoder và revision mà nhóm đã khóa (`clip_encoder.json` / `text_encoder.json`),
xếp hạng CLIP trên toàn bộ pool test, lấy top-1000 bằng sắp xếp ổn định, rồi rerank bằng điểm AdaMF với α đã khóa.
So sánh từng truy vấn với `test_query_ranks.csv` của nhóm. Phần CLIP giống nhau giữa các seed nên chỉ tính một lần.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 5, globals())


Cột `khop_*` là tỉ lệ truy vấn có thứ hạng trùng tuyệt đối với file của nhóm. Audit DB15K (09/11) đã ghi nhận
10 truy vấn có CLIP rank lệch ±1 giữa các run của nhóm (F02), nên tỉ lệ dưới 100% một chút ở cột CLIP là có thể;
nếu thấp hơn nhiều thì dừng lại kiểm tra encoder/revision trước khi đọc tiếp.

## C2. Đồ thị cứu và làm hại bao nhiêu truy vấn

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 8, globals())


## C3. Bộ 20 truy vấn đánh giá mỗi tập (10 trực tiếp + 5 head + 5 tail)

Luật cố định, không nhìn kết quả: lấy từ test, bỏ truy vấn cold-start (thực thể đích không có trong train),
mỗi thực thể đích chỉ một truy vấn, bốc ngẫu nhiên với `SELECTION_SEED = 2026`. Bộ này chỉ dùng để đánh giá
và minh họa giải thích, không dùng tinh chỉnh gì. Báo cáo tách nhóm trực tiếp / gián tiếp, trung bình ± độ lệch chuẩn 3 seed.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 10, globals())


## C4–C5. Bộ giải thích

Giải thích một ảnh `img` được xếp ở hạng nào cho truy vấn `q` (chỉ áp dụng khi điểm đồ thị tham gia xếp hạng:
truy vấn gián tiếp ở cả 2 tập, và truy vấn trực tiếp ở FB15K-237).

**E1 — thực thể quyết định.** Điểm đồ thị của ảnh là max qua các thực thể gắn với ảnh; thực thể đạt max là thực thể
quyết định (lấy thẳng argmax của hàm chấm điểm nên đúng tuyệt đối).

**E2 — đóng góp từng phương thức.** `joint(e) = Σ_m w_m·v_m` với m ∈ {cấu trúc, ảnh, văn bản}. Trọng số attention `w_m`
là thông tin nội tại nhưng chưa đủ (attention không phải giải thích). Vì vậy đo thêm **Δ khi che phương thức m** của
thực thể đó: đặt cờ có mặt = False đúng như cơ chế mask mà mô hình dùng cho phương thức thiếu, tính lại điểm và hạng.

**E3 — bằng chứng từ đồ thị.** Embedding cấu trúc của `e` được học từ các bộ ba train chứa `e`. Để biết bộ ba nào
thật sự làm mô hình tin `(anchor, r, e)`, dùng ý tưởng *post-training* của Kelpie (Rossi và cs., SIGMOD 2022): học lại
riêng embedding cấu trúc của `e` (mọi tham số khác giữ nguyên) từ các bộ ba của nó, một lần đủ bộ ba và các lần bỏ bớt;
chênh lệch điểm giữa hai bản học lại là đóng góp của các bộ ba bị bỏ. Mọi biến thể dùng cùng khởi tạo, cùng mẫu âm,
cùng số bước nên chênh lệch chỉ do bộ ba bị bỏ. Δ này được trừ vào **điểm gốc** của mô hình để tính hạng mới, nên hạng
luôn đo trên chính checkpoint của nhóm; độ khớp giữa bản học lại đủ bộ ba và điểm gốc được báo cáo ở C6. Bộ ba bằng chứng được nối thành **đường đi từ anchor** khi có:
`anchor –r'– e` (1 bước) hoặc `anchor –r''– x –r'– e` (2 bước, qua bộ ba train khác).

Chỉ dùng bộ ba **train**; bộ ba của truy vấn test không có trong train (split sạch), nên giải thích không thể là đáp án.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 12, globals())


## C6. Chọn số bộ ba K trên validation

Giải thích ảnh hạng 1 mà hệ thống trả về cho `N_VAL_K` truy vấn gián tiếp validation (seed chính). Với mỗi K,
độ đầy đủ = (Δ khi bỏ đồng thời top-K) / (Δ khi bỏ cả M bộ ba ứng viên). Chọn K nhỏ nhất có trung vị ≥ 0.8.
Đồng thời báo cáo độ khớp của bản học lại đủ bộ ba với điểm gốc (học lại có tái tạo được mô hình không).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 14, globals())


## C7. Chọn ca theo luật cố định (seed chính, mỗi tập 5 thành công + 5 thất bại)

- **Thành công**: truy vấn gián tiếp, CLIP xếp ảnh đúng ≥ 20, CLIP + AdaMF đưa lên hạng 1.
- **Thất bại – bị làm hại**: CLIP xếp ảnh đúng ≤ 5 nhưng sau rerank tụt hạng (3 ca).
- **Thất bại – không cứu được**: ảnh đúng nằm trong top-1000 nhưng vẫn ở hạng ≥ 100 sau rerank (2 ca).

Bốc ngẫu nhiên với `SELECTION_SEED`, mỗi ca khác quan hệ và khác thực thể đích. Thiếu ca ở nhóm nào thì bù từ nhóm còn lại.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 16, globals())


## C8. Giải thích và hình cho từng ca

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 18, globals())


## C9. Kiểm chứng giải thích đến từ cơ chế xếp hạng (3 seed)

Với `N_FID` truy vấn gián tiếp test mỗi seed (bốc cố định), giải thích **ảnh hạng 1 mà hệ thống trả về**, rồi:
- E3: bỏ đồng thời top-K* bộ ba bằng chứng so với bỏ K* bộ ba ngẫu nhiên của cùng thực thể (`N_RANDOM` lần);
- E2: che phương thức cấu trúc (phần do đồ thị học ra) và xem ảnh còn giữ hạng 1 không.

Giải thích trung thực thì bỏ bằng chứng phải làm điểm giảm và hạng tụt rõ hơn hẳn bỏ ngẫu nhiên.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 20, globals())


## C10. Demo: truy vấn → top-k ảnh + đồ thị giải thích tương tác

Giao diện dùng chung cho cả ba cấp (`multimodal_xai/demo_web.py`). Chạy riêng demo: chạy ô C0, C0b, C0c rồi chạy thẳng hai ô
dưới (K* và 10 ca đọc từ kết quả C6, C7 đã lưu); hoặc mở `demo/demo_cap2_db15k_fb15k.ipynb`.
- **Ca trong báo cáo**: chọn một trong 10 ca (5 thành công, 5 thất bại) là trang tự chạy, có ✓ ở ảnh đúng và dòng
  "Ảnh đúng: CLIP hạng a → CLIP + AdaMF hạng b".
- **Tự nhập truy vấn**: gián tiếp (thực thể neo + quan hệ + chiều, loại truy vấn mà đồ thị tham gia xếp hạng) hoặc trực tiếp
  (câu tự do; DB15K theo thiết kế chỉ dùng CLIP nên không có giải thích đồ thị).

Đồ thị trả lời "Vì sao trả về X?": nút xanh ngọc = neo, nút cam = ảnh trả về (vẽ bằng chính ảnh), cạnh xanh dương = bộ ba
giải thích S (dày theo Δ, số trong ngoặc là Δ), cạnh chấm xanh nhạt = bộ ba nối về neo (không đo), cạnh xám đứt = quan hệ được hỏi.
Nút **Bỏ giải thích S** cho thấy hạng của ảnh sau khi bỏ S, so với bỏ cùng số bộ ba ngẫu nhiên (C9).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 22, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('07_db15k_fb15k_adamf_xai_level2', 23, globals())
